In [3]:
pip install SimpleITK

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 14.0 MB/s eta 0:00:00


In [1]:
# ==============================================================================
# EVALUATION : 3D DIFFUSION MODEL PERFORMANCE
# ==============================================================================

In [4]:
#Environment Setup & Dependencies
import os
import torch
import numpy as np
import SimpleITK as sitk
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

root_dir = "/content/drive/MyDrive/.../dirlab"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [5]:
#Load the Model Checkpoint
CHECKPOINT_PATH = os.path.join(root_dir, "checkpoints", "best_model.pt")

# TODO: Import your model class from your training script
# from diffusion_training import YourModelClass

# model = YourModelClass().to(device)
# model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
# model.eval()
print("Model loaded into memory and set to evaluation mode.")

Model loaded into memory and set to evaluation mode.


### Define Custom Dataset and DataLoader

To create `val_loader`, we need a PyTorch `Dataset` that can load your data. This example assumes you have a `validation_data` directory within your `root_dir`, containing files that can be read by SimpleITK. You will need to adapt the `CustomDataset` class to match your specific data structure and how your 'target' and 'input' volumes are organized.

In [8]:
from torch.utils.data import Dataset, DataLoader

class CustomDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_paths = []
        # Example: Assuming validation data is in a subfolder named 'validation_data'
        validation_dir = os.path.join(root_dir, 'validation_data')

        if not os.path.exists(validation_dir):
            print(f"Warning: Validation data directory not found at {validation_dir}. Please create it or adjust the path in CustomDataset.")
        else:
            # Populate image_paths with paths to your data files
            # This is a placeholder; you'll need to adapt it to how your files are named/organized.
            # For example, if you have 'case001_input.mha' and 'case001_target.mha'
            for filename in os.listdir(validation_dir):
                if filename.endswith('_target.mha'): # Or whatever your target file extension is
                    self.image_paths.append(os.path.join(validation_dir, filename))

            if not self.image_paths:
                print(f"Warning: No image files found in {validation_dir}. Please ensure your data files are present and match the expected naming convention.")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        target_path = self.image_paths[idx]

        # Assuming input and target files are related by name, e.g., 'caseXXX_target.mha' and 'caseXXX_input.mha'
        input_path = target_path.replace('_target.mha', '_input.mha') # Adjust this based on your naming convention

        # Load images using SimpleITK
        target_itk = sitk.ReadImage(target_path)
        input_itk = sitk.ReadImage(input_path)

        # Convert SimpleITK image to NumPy array and then to PyTorch tensor
        target_np = sitk.GetArrayFromImage(target_itk).astype(np.float32)
        input_np = sitk.GetArrayFromImage(input_itk).astype(np.float32)

        # Normalize data (assuming values are between 0 and 255 and need to be in [-1, 1])
        # Adjust normalization based on your actual data range and model requirements
        target_tensor = (torch.from_numpy(target_np) / 127.5) - 1.0
        input_tensor = (torch.from_numpy(input_np) / 127.5) - 1.0

        # Add a channel dimension if your model expects it (e.g., [C, D, H, W])
        target_tensor = target_tensor.unsqueeze(0)
        input_tensor = input_tensor.unsqueeze(0)

        sample = {'input': input_tensor, 'target': target_tensor}

        if self.transform:
            sample = self.transform(sample)

        return sample

# Instantiate the dataset and DataLoader
# You might want to define a specific batch size
batch_size = 1

val_dataset = CustomDataset(root_dir=root_dir) # You can add transforms here if needed
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"val_loader created with {len(val_dataset)} samples and batch size {batch_size}.")


val_loader created with 0 samples and batch size 1.


In [6]:
#Define Metrics & Helper Functions
def calculate_metrics(real_volume, generated_volume):
    """Calculates MAE, PSNR, and 3D SSIM between two normalized volumes."""
    real = real_volume.detach().cpu().numpy().squeeze()
    gen = generated_volume.detach().cpu().numpy().squeeze()

    mae_val = np.mean(np.abs(real - gen))
    data_range = 2.0  # Data is normalized between [-1, 1]
    psnr_val = psnr(real, gen, data_range=data_range)
    ssim_val = ssim(real, gen, data_range=data_range, channel_axis=None)

    return mae_val, psnr_val, ssim_val

In [9]:
#The Evaluation Loop & Final Results
mae_scores, psnr_scores, ssim_scores = [], [], []

# Assuming val_loader is defined/imported
with torch.no_grad():
    for batch_idx, batch in enumerate(val_loader):
        real_target = batch['target'].to(device)
        conditioned_input = batch['input'].to(device)

        # generated_output = ddim_sampler(model, conditioned_input, steps=50)
        generated_output = real_target + torch.randn_like(real_target) * 0.05 # placeholder

        mae, p_snr, s_sim = calculate_metrics(real_target, generated_output)
        mae_scores.append(mae)
        psnr_scores.append(p_snr)
        ssim_scores.append(s_sim)

        print(f"Case {batch_idx+1} -> MAE: {mae:.4f} | PSNR: {p_snr:.2f} dB | SSIM: {s_sim:.4f}")

print("\n" + "="*40 + "\nFINAL EVALUATION METRICS\n" + "="*40)
print(f"Mean Absolute Error (MAE):  {np.mean(mae_scores):.4f} ± {np.std(mae_scores):.4f}")
print(f"Peak Signal-to-Noise Ratio: {np.mean(psnr_scores):.2f} ± {np.std(psnr_scores):.2f} dB")
print(f"Structural Similarity (SSIM): {np.mean(ssim_scores):.4f} ± {np.std(ssim_scores):.4f}")


FINAL EVALUATION METRICS
Mean Absolute Error (MAE):  nan ± nan
Peak Signal-to-Noise Ratio: nan ± nan dB
Structural Similarity (SSIM): nan ± nan


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:218: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:175: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:210: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
